In [150]:
import pandas as pd
import numpy as np
from alerce.core import Alerce
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import SkyCoord

alerce = Alerce()

In [151]:
tns = pd.read_csv("tns_search.csv")
column_names = tns.columns.tolist()
print(column_names)

ztf = pd.read_csv("ztf_bts_explorer.csv")
column_names = ztf.columns.tolist()
print(column_names)


['ID', 'Name', 'RA', 'DEC', 'Obj. Type', 'Redshift', 'Host Name', 'Host Redshift', 'Reporting Group/s', 'Discovery Data Source/s', 'Classifying Group/s', 'Associated Group/s', 'Disc. Internal Name', 'Disc. Instrument/s', 'Class. Instrument/s', 'TNS AT', 'Public', 'End Prop. Period', 'Discovery Mag/Flux', 'Discovery Filter', 'Discovery Date (UT)', 'Sender', 'Remarks', 'Unreal', 'Discovery Bibcode', 'Classification Bibcodes', 'Ext. catalog/s', 'Auto classification']
['ZTFID', 'IAUID', 'RA', 'Dec', 'peakt', 'peakfilt', 'peakmag', 'peakabs', 'duration', 'rise', 'fade', 'type', 'redshift', 'b', 'A_V']


In [152]:
def convert_ra(ra_coords):
    ra_new = SkyCoord(ra_coords)
    
    return ra_new


In [153]:
def convert_dec(dec_coords):
    dec_coords = str(dec_coords)
    
    degrees, mins, secs = dec_coords.split(":")

    if float(degrees) < 0:
        dec_new = (float(degrees) - float(mins) / 60 - float(mins) / 3600)
        
    else:
        dec_new = (float(degrees) + float(mins) / 60 + float(secs) / 3600)

    return dec_new


In [154]:
def convert_mjd(times):
    t = Time(times, format='iso', scale='utc')
    
    return t.mjd


In [155]:
def convert_coords(tns_ra_coords, tns_dec_coords):
    new_vals = SkyCoord(tns_ra_coords, tns_dec_coords, unit = (u.hourangle, u.deg))

    return new_vals


In [156]:
df_tns_coords = convert_coords(tns['RA'].values, tns['DEC'].values)
dt_ztf_coords = convert_coords(ztf['RA'].values, ztf['Dec'].values)

tns_ra = df_tns_coords.ra
tns_dec = df_tns_coords.dec

ztf_ra = dt_ztf_coords.ra
ztf_dec = dt_ztf_coords.dec

tns_coords = SkyCoord(ra=tns_ra,dec=tns_dec)
ztf_coords = SkyCoord(ra=ztf_ra, dec=ztf_dec)

idx, d2d, d3d = tns_coords.match_to_catalog_sky(ztf_coords) # Look at TNS and find the closest ZTF
# idx = index of closest ZTF object to each TNS object 
# d2d = angular distance in the sky between the TNS object and the closest ZTF object




In [157]:
max_sep = 1.0 * u.arcsec
sep_constraint = d2d < max_sep
matches = tns_coords[sep_constraint]

tns['match_index'] = idx
tns['distance'] = d2d

tns_matched_df = tns.iloc[sep_constraint].copy()

ztf_sub = ztf[ztf.index.isin(tns_matched_df['match_index'].values)]

ztf2tns = ztf_sub.merge(tns_matched_df,
                        left_index = True,
                        right_on = 'match_index',
                        how = 'inner'
)

ztf2tns.to_csv('ztf_tns_matches_1arcsec.csv', index=False)

In [158]:
max_sep = 5.0 * u.arcsec
sep_constraint = d2d < max_sep
matches = tns_coords[sep_constraint]

tns['match_index'] = idx
tns['distance'] = d2d

tns_matched_df = tns.iloc[sep_constraint].copy()

ztf_sub = ztf[ztf.index.isin(tns_matched_df['match_index'].values)]

ztf2tns = ztf_sub.merge(tns_matched_df,
                        left_index = True,
                        right_on = 'match_index',
                        how = 'inner'
)

ztf2tns.to_csv('ztf_tns_matches_5arcsec.csv', index=False)

In [159]:
max_sep = 10.0 * u.arcsec
sep_constraint = d2d < max_sep
matches = tns_coords[sep_constraint]

tns['match_index'] = idx
tns['distance'] = d2d

tns_matched_df = tns.iloc[sep_constraint].copy()

ztf_sub = ztf[ztf.index.isin(tns_matched_df['match_index'].values)]

ztf2tns = ztf_sub.merge(tns_matched_df,
                        left_index = True,
                        right_on = 'match_index',
                        how = 'inner'
)

ztf2tns.to_csv('ztf_tns_matches_10arcsec.csv', index=False)

In [160]:
max_sep = 20.0 * u.arcsec
sep_constraint = d2d < max_sep
matches = tns_coords[sep_constraint]

tns['match_index'] = idx
tns['distance'] = d2d

tns_matched_df = tns.iloc[sep_constraint].copy()

ztf_sub = ztf[ztf.index.isin(tns_matched_df['match_index'].values)]

ztf2tns = ztf_sub.merge(tns_matched_df,
                        left_index = True,
                        right_on = 'match_index',
                        how = 'inner'
)

ztf2tns.to_csv('ztf_tns_matches_20arcsec.csv', index=False)

In [161]:
# Get a CSV of the Unmatched TNS Objects
max_sep = 20.0 * u.arcsec
sep_constraint = d2d >= max_sep

tns['match_index'] = idx
tns['distance'] = d2d

tns_unmatched_df = tns.iloc[sep_constraint].copy()

tns_unmatched_df.to_csv('ztf_tns_unmatched.csv', index=False)

In [162]:
overlapping = set(tns_matched_df['Name']) & set(tns_unmatched_df['Name'])

print(overlapping)
print(len(overlapping))

set()
0


In [168]:
supernovae = tns[~tns["Host Redshift"].isna()]
supernovae

,ID,Name,RA,DEC,Obj. Type,Redshift,Host Name,Host Redshift,Reporting Group/s,Discovery Data Source/s,...,Discovery Date (UT),Sender,Remarks,Unreal,Discovery Bibcode,Classification Bibcodes,Ext. catalog/s,Auto classification,match_index,distance
33,194416,SN 2025afzk,23:14:43.352,+31:19:09.60,SN IIb,0.023390,PGC070793,0.023400,"SNHunt, GOTO, ATLAS, Pan-STARRS, ZTF","CRTS, GOTO, ATLAS, Pan-STARRS, ZTF",...,2025-12-08 03:57:57.000,Andrew Drake,[Object cross-matched with PGC070793],NaN,2025TNSTR4879....1M,2025TNSCR5025....1S,NaN,NaN,64,6.424380
57,179421,SN 2025kkv,09:59:46.167,-06:44:26.28,SN IIb,0.016000,PGC1029331,0.016235,"SGLF, BlackGEM, ATLAS","ZTF, BlackGEM, ATLAS",...,2025-05-15 04:09:14.000,Ismael Perez-Fournon,[Object cross-matched with PGC1029331],NaN,2025TNSTR1793....1P,2025TNSCR2079....1D,NaN,NaN,11,14.824653
73,167461,SN 2024acsi,14:30:52.520,+12:36:52.78,SN IIb,0.026320,PGC086478,0.026324,"XOSS, ZTF, GOTO, GaiaAlerts, ATLAS, YSE, Pan-S...","XOSS, ZTF, GOTO, GaiaAlerts, ATLAS, Pan-STARRS...",...,2024-11-29 23:16:56.640,Mi Zhang,[Object cross-matched with PGC086478],NaN,2024TNSTR4697....1Z,2024TNSCR4956....1T,NaN,NaN,83,6.142008
91,153671,SN 2024ljc,19:01:41.420,+40:44:42.24,SN IIb,0.015154,NGC6745,0.015158,"SGLF, ATLAS, Pan-STARRS","ZTF, ATLAS, Pan-STARRS",...,2024-06-15 08:15:05.000,Ismael Perez-Fournon,[Object cross-matched with NGC6745],NaN,2024TNSTR1967....1P,"2024TNSCR1987....1P, 2024TNSCR2147....1W, 2024...",NaN,NaN,86,0.000039
94,151538,SN 2024ixe,22:34:35.501,+05:49:11.18,SN IIb,0.027609,NGC7312,0.027600,"SNHunt, ATLAS, ZTF, Pan-STARRS","CRTS, ATLAS, ZTF, Pan-STARRS",...,2024-05-15 11:15:17.000,Andrew Drake,[Object cross-matched with NGC7312],NaN,2024TNSTR1529....1M,2024TNSCR2734....1Q,NaN,NaN,55,10.127687
96,150181,SN 2024gzk,11:28:32.397,+58:33:44.08,SN IIb,0.010300,NGC3690,0.010300,NUTS2,ZTF,...,2024-04-12 08:23:04.000,Erkki Kankare,[Object cross-matched with NGC3690],NaN,2024TNSTR1159....1K,2024TNSCR1169....1K,NaN,NaN,18,0.000079
98,149453,SN 2024fzj,17:02:25.394,+31:13:07.02,SN IIb,0.034000,PGC084819,0.033778,"ALeRCE, ATLAS, Pan-STARRS, YSE","ZTF, ATLAS, Pan-STARRS",...,2024-04-04 10:20:46.997,ALeRCE,[Object cross-matched with PGC084819],NaN,2024TNSTR.971....1M,2024TNSCR2572....1D,NaN,NaN,82,0.000040
116,138136,SN 2023uvh,09:07:55.169,+03:16:39.27,SN IIb,0.027000,PGC025666,0.026760,"ALeRCE, GOTO, YSE, Pan-STARRS","ZTF, GOTO, Pan-STARRS",...,2023-10-13 11:03:39.004,ALeRCE,[Object cross-matched with PGC025666],NaN,2023TNSTR2574....1F,2023TNSCR2787....1T,NaN,NaN,77,0.000042
121,132407,SN 2023mut,04:48:34.348,+00:14:48.31,SN IIb,0.002200,UGC 03174,0.002238,"ASAS-SN, ATLAS, ZTF, Pan-STARRS","ASAS-SN, ATLAS, ZTF, Pan-STARRS",...,2023-07-11 04:19:12.000,Krzysztof Stanek,NaN,NaN,2023TNSTR1626....1S,2023TNSCR1699....1T,NaN,NaN,68,3.909586
129,126289,SN 2023eaf,10:11:51.235,+58:53:28.84,SN IIb,0.030400,PGC029697,0.030458,"ALeRCE, ATLAS, Pan-STARRS","ZTF, ATLAS, Pan-STARRS",...,2023-03-27 08:24:05.996,ALeRCE,[Object cross-matched with PGC029697],NaN,2023TNSTR.639....1M,2023TNSCR.855....1D,NaN,NaN,72,0.000015
